# Huấn luyện CNN nhận diện động vật

Notebook này là phiên bản trình bày của quy trình huấn luyện mô hình `animal_image_classifier.keras`. Mô hình chỉ phân loại các lớp trong tập dữ liệu của dự án; không sử dụng mô hình hay nhãn dự phòng bên ngoài.

## 1. Thiết lập môi trường và siêu tham số

Ta xây dựng CNN từ đầu để có thể trình bày rõ vai trò của từng lớp convolution, pooling, normalization và classifier.

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VALIDATION_SPLIT = 0.2
EPOCHS = 20
DATA_DIR = Path('dataset_raw/animals/animals')
MODEL_PATH = Path('animal_image_classifier.keras')

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
assert DATA_DIR.exists(), f'Không tìm thấy dữ liệu: {DATA_DIR.resolve()}'

## 2. Preprocessing: đưa mọi ảnh về cùng đầu vào

Preprocessing là **xác định và bắt buộc** ở cả train lẫn inference; nó khác augmentation vốn chỉ áp dụng khi train. Các bước là: (1) mở ảnh và chuyển RGB 3 kênh, (2) resize 224×224 bằng bilinear, (3) đổi sang `float32`, (4) thêm batch dimension thành `(1, 224, 224, 3)`. Chưa chia 255 ở đây vì layer `Rescaling` là layer đầu tiên của CNN, nên train và API luôn chuẩn hoá giống nhau.

In [ ]:
def preprocess_for_inference(image: Image.Image) -> np.ndarray:
    """Tiền xử lý một ảnh upload; giống hệt model/preprocess.py."""
    image = image.convert('RGB').resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
    image_array = np.asarray(image, dtype=np.float32)
    return np.expand_dims(image_array, axis=0)

# Ví dụ khi có một ảnh upload:
# batch = preprocess_for_inference(Image.open('duong_dan_anh.jpg'))
# print(batch.shape, batch.dtype, batch.min(), batch.max())  # (1, 224, 224, 3), float32, 0..255

**Áp dụng cho dự án:** frontend gửi ảnh bất kỳ từ camera hoặc file upload đến FastAPI. `model/preprocess.py` thực thi đúng hàm trên trước `model.predict()`. Nếu bỏ bước RGB, ảnh PNG có alpha hoặc ảnh xám có thể sai số kênh; nếu bỏ resize, CNN không nhận đúng input shape; nếu thay đổi cách scale ở một phía, kết quả web sẽ khác khi đánh giá trong notebook.

## 3. Đọc và kiểm tra dữ liệu

`image_dataset_from_directory` gán chỉ số nhãn theo thứ tự tên thư mục. Khi train, hàm này decode ảnh RGB và resize từng ảnh theo `IMAGE_SIZE`. Danh sách `class_names` phải được lưu lại vì API dùng nó để đổi kết quả softmax thành tên loài.

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VALIDATION_SPLIT, subset='training', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode='int'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR, validation_split=VALIDATION_SPLIT, subset='validation', seed=SEED,
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, label_mode='int'
)

class_names = train_ds.class_names
assert class_names == val_ds.class_names
print(f'Số lớp: {len(class_names)}')
print(class_names)

In [ ]:
images, labels = next(iter(train_ds.take(1)))
plt.figure(figsize=(12, 8))
for index in range(min(12, len(images))):
    ax = plt.subplot(3, 4, index + 1)
    ax.imshow(images[index].numpy().astype('uint8'))
    ax.set_title(class_names[int(labels[index])])
    ax.axis('off')
plt.tight_layout()

## 4. Tăng cường dữ liệu và luồng xử lý

Augmentation chỉ bật khi train để mô hình bền vững hơn với ảnh lật, xoay nhẹ và thay đổi độ tương phản. Lớp `Rescaling` trong CNN sẽ chuẩn hoá pixel từ [0, 255] về [0, 1].

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name='data_augmentation')

## 5. Xây dựng kiến trúc CNN — từng lớp

Luồng kích thước tensor:

| Giai đoạn | Các lớp | Kích thước đầu ra |
|---|---|---|
| Input | Input + Rescaling | 224×224×3 |
| Block 1 | Conv2D(32, 3×3) + BatchNorm + MaxPool | 112×112×32 |
| Block 2 | Conv2D(64, 3×3) + BatchNorm + MaxPool | 56×56×64 |
| Block 3 | Conv2D(128, 3×3) + BatchNorm + MaxPool | 28×28×128 |
| Block 4 | Conv2D(256, 3×3) + BatchNorm + MaxPool | 14×14×256 |
| Classifier | GlobalAveragePooling + Dense(256) + Dropout + Softmax | 47 xác suất |

### 5.1 Bản đồ từng layer của mô hình

| # | Layer trong code | Output shape | Ý nghĩa CNN nói chung | Áp dụng trong Animal Explorer |
|---:|---|---|---|---|
| 1 | `image` | 224×224×3 | Nhận tensor RGB đầu vào. | Ảnh người dùng upload được resize đồng nhất để mô hình xử lý. |
| 2 | `data_augmentation` | 224×224×3 | Tạo biến thể ảnh trong lúc train, không tạo tham số học. | Mô phỏng động vật quay trái/phải, xa/gần, ánh sáng khác nhau khi người dùng chụp ảnh. |
| 3 | `rescaling` | 224×224×3 | Chia pixel 255 để đưa dữ liệu về [0, 1], giúp gradient ổn định. | Camera, file PNG/JPG có cường độ pixel khác nhau nhưng được đưa về cùng thang đo. |
| 4 | `conv_1` + ReLU | 224×224×32 | 32 kernel 3×3 quét ảnh để tìm cạnh, đường cong, tương phản và texture đơn giản. | Có thể phản ứng với viền cánh chim, mắt, chân, vảy hoặc rìa mai rùa. |
| 5 | `batch_norm_1` | 224×224×32 | Chuẩn hoá output mini-batch để train nhanh và ít nhạy với khởi tạo. | Giảm ảnh hưởng khi dữ liệu có ảnh nền sáng/tối hoặc chất lượng khác nhau. |
| 6 | `max_pool_1` | 112×112×32 | Lấy giá trị mạnh nhất trong mỗi vùng 2×2, giảm một nửa chiều rộng/cao. | Giữ tín hiệu như cạnh mắt/tai dù vị trí lệch vài pixel. |
| 7 | `conv_2` + ReLU | 112×112×64 | Ghép các cạnh thành mẫu cục bộ; tăng lên 64 feature maps. | Học mỏ, tai, đốm lông, sọc ngựa vằn, hoa văn mai hoặc vây. |
| 8 | `batch_norm_2` | 112×112×64 | Ổn định phân phối 64 đặc trưng của block 2. | Hữu ích khi cùng một loài xuất hiện ở môi trường rừng, nước hoặc nền nhân tạo. |
| 9 | `max_pool_2` | 56×56×64 | Nén đặc trưng cấp thấp để tạo receptive field lớn hơn. | Mô hình bắt đầu quan tâm một vùng cơ thể thay vì một điểm ảnh đơn lẻ. |
| 10 | `conv_3` + ReLU | 56×56×128 | 128 kernel học tổ hợp đặc trưng trung cấp. | Kết hợp mắt + mỏ + lông để nhận biết chim; vảy + đầu + thân để phân biệt rắn/cá sấu. |
| 11 | `batch_norm_3` | 56×56×128 | Giữ quá trình cập nhật trọng số ổn định ở tầng sâu hơn. | Tránh mô hình thiên lệch quá mạnh theo màu nền thay vì đặc điểm con vật. |
| 12 | `max_pool_3` | 28×28×128 | Giảm kích thước, ưu tiên sự hiện diện của đặc trưng hơn tọa độ chính xác. | Vẫn nhận ra động vật khi không nằm chính giữa khung hình. |
| 13 | `conv_4` + ReLU | 28×28×256 | 256 kernel học đặc trưng cấp cao, gần với khái niệm đối tượng. | Phân biệt các lớp gần nhau như `african_crocodile`/`siamese_crocodile`, `duck`/`goose`, `turtle`/`elongated_tortoise`. |
| 14 | `batch_norm_4` | 28×28×256 | Ổn định representation giàu đặc trưng trước classifier. | Giúp quyết định cuối dựa trên tổ hợp dấu hiệu của loài thay vì một chi tiết ngẫu nhiên. |
| 15 | `max_pool_4` | 14×14×256 | Tạo feature map gọn nhưng vẫn giữ 256 kênh ngữ nghĩa. | Mỗi vùng map lúc này đại diện cho một vùng lớn trên cơ thể con vật. |
| 16 | `global_average_pooling` | 256 | Lấy trung bình 14×14 của mỗi kênh, biến feature map thành vector 256 chiều. | Tóm tắt mức độ xuất hiện của 256 dấu hiệu về loài, đồng thời ít phụ thuộc vị trí con vật. |
| 17 | `dense_features` + ReLU | 256 | Học quan hệ phi tuyến giữa các đặc trưng đã tóm tắt. | Ghép các dấu hiệu như “mỏ cong + cánh rộng” hoặc “thân dài + vảy” trước khi gọi tên loài. |
| 18 | `dropout` | 256 | Ngẫu nhiên tắt 40% neuron chỉ khi train để giảm overfitting. | Hạn chế việc nhớ các bối cảnh lặp lại trong dataset, ví dụ nền nước hay nền rừng. |
| 19 | `classifier` + Softmax | 47 | Dense trả logit cho mỗi lớp; Softmax chuyển chúng thành tổng xác suất bằng 1. | API chọn xác suất lớn nhất để trả một trong 47 nhãn tại `labels.py`, ví dụ `whale`, `pangolin`, `owl`. |

### 5.2 Công thức ngắn để giải thích khi bảo vệ

- **Convolution:** với filter k, `z[i,j,k] = Σ W[u,v,c,k] × x[i+u,j+v,c] + b[k]`; layer `Conv2D` học các trọng số W này từ ảnh động vật, không phải quy tắc viết tay. `padding='same'` giữ nguyên H×W trước pooling.
- **ReLU:** `max(0, z)` loại bỏ tín hiệu âm, tạo tính phi tuyến để 4 block có thể học hình dạng phức tạp thay vì chỉ tổ hợp tuyến tính của pixel.
- **Batch normalization:** chuẩn hoá activation theo batch rồi học lại hệ số scale/shift; nhờ đó learning rate 1e-3 ổn định hơn.
- **Max pooling:** `max` trên ô 2×2; vì vậy output lần lượt giảm `224 → 112 → 56 → 28 → 14`, giảm chi phí tính toán và tăng vùng ảnh mà tầng sâu quan sát.
- **Softmax:** `p(lớp k) = exp(z_k) / Σ exp(z_j)`. Giá trị API trả về là `max(p)`, nhưng nếu thấp thì UI phải cảnh báo vì ảnh có thể mờ hoặc nằm ngoài 47 lớp đã huấn luyện.

**Điểm cần nói rõ:** CNN chỉ dự đoán trong 47 loài của `labels.py`; hệ thống hiện không dùng bất kỳ fallback nào để thay nhãn bằng một mô hình khác.

In [ ]:
inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,), name='image')
x = data_augmentation(inputs)
x = tf.keras.layers.Rescaling(1.0 / 255, name='rescaling')(x)

# Block 1: cạnh và texture cơ bản
x = tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu', name='conv_1')(x)
x = tf.keras.layers.BatchNormalization(name='batch_norm_1')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=2, name='max_pool_1')(x)

# Block 2: các mẫu cục bộ như mắt, tai, vân lông
x = tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu', name='conv_2')(x)
x = tf.keras.layers.BatchNormalization(name='batch_norm_2')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=2, name='max_pool_2')(x)

# Block 3: đặc trưng hình dạng cấp cao hơn
x = tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu', name='conv_3')(x)
x = tf.keras.layers.BatchNormalization(name='batch_norm_3')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=2, name='max_pool_3')(x)

# Block 4: kết hợp đặc trưng để phân biệt loài
x = tf.keras.layers.Conv2D(256, 3, padding='same', activation='relu', name='conv_4')(x)
x = tf.keras.layers.BatchNormalization(name='batch_norm_4')(x)
x = tf.keras.layers.MaxPooling2D(pool_size=2, name='max_pool_4')(x)

x = tf.keras.layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = tf.keras.layers.Dense(256, activation='relu', name='dense_features')(x)
x = tf.keras.layers.Dropout(0.40, name='dropout')(x)
outputs = tf.keras.layers.Dense(len(class_names), activation='softmax', name='classifier')(x)
model = tf.keras.Model(inputs, outputs, name='animal_cnn')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
# Bảng chi tiết: dùng trực tiếp khi thuyết trình từng layer
for index, layer in enumerate(model.layers, start=1):
    print(f'{index:>2}. {layer.name:<24} {layer.__class__.__name__:<24} -> {layer.output.shape}')

## 6. Huấn luyện

`EarlyStopping` dừng khi validation loss không còn cải thiện; `ModelCheckpoint` luôn lưu phiên bản tốt nhất theo validation loss.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor='val_loss', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2),
]

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks
)

## 7. Theo dõi loss và accuracy

Khoảng cách lớn giữa đường train và validation là tín hiệu overfitting cần nêu khi trình bày.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Loss')
axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
for ax in axes: ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()

## 8. Đánh giá trên tập validation

Accuracy cho biết tỷ lệ dự đoán đúng tổng thể; ma trận nhầm lẫn giúp giải thích những cặp loài mà CNN dễ nhầm.

In [ ]:
val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f'Validation loss: {val_loss:.4f}')
print(f'Validation accuracy: {val_accuracy:.2%}')

y_true = np.concatenate([y.numpy() for _, y in val_ds])
y_pred = np.argmax(model.predict(val_ds, verbose=0), axis=1)
cm = tf.math.confusion_matrix(y_true, y_pred, num_classes=len(class_names)).numpy()
plt.figure(figsize=(10, 8))
plt.imshow(cm, cmap='Blues')
plt.title('Confusion matrix (validation)')
plt.xlabel('Nhãn dự đoán')
plt.ylabel('Nhãn thật')
plt.colorbar()
plt.show()

## 9. Lưu model và danh sách nhãn

Hai tệp này được FastAPI sử dụng khi chạy ứng dụng. Lưu `labels.py` cùng thứ tự với đầu ra Softmax là điều bắt buộc.

In [ ]:
model.save(MODEL_PATH)
Path('labels.py').write_text(
    '# -*- coding: utf-8 -*-\nclasses = ' + repr(class_names) + '\n',
    encoding='utf-8',
)
print(f'Đã lưu model: {MODEL_PATH.resolve()}')
print('Đã cập nhật labels.py')

## 10. Thông điệp trình bày

- CNN trích xuất đặc trưng không gian bằng convolution; các tầng sâu nhận biết hình dạng phức tạp hơn.
- Các block convolution học đặc trưng theo cấp bậc: cạnh/texture → bộ phận → hình dạng loài.
- Softmax trả về xác suất trên đúng các lớp đã huấn luyện; vì vậy ảnh ngoài tập lớp vẫn có thể có độ tin cậy thấp và cần được thông báo rõ cho người dùng.
- Grad-CAM trong ứng dụng trực quan hoá vùng ảnh đóng góp nhiều nhất cho lớp CNN dự đoán.